[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_07/notebook_7_4_ecg_classification_1d_cnn.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.4: ECG Signal Classification for AFib Detection

**Chapter 7: Time Series Analysis - Implementing Journey 2 (Yuki)**

**Journey Connection**: This notebook implements Journey 2 from Chapter 3, where Yuki received multiple false AFib alerts from a wearable device. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

1. Preprocess ECG waveforms (filtering, quality assessment)
2. Extract hand-crafted features (R-R intervals, HRV)
3. Train 1D CNN for rhythm classification
4. Analyze false positives and motion artifacts
5. Understand the impact of prevalence on PPV

## Clinical Context

Atrial fibrillation increases stroke risk 5×. Wearable ECG devices can catch paroxysmal AFib missed in office visits.

**Yuki's story**: Smartwatch alerted to irregular rhythm. Multiple false alarms from motion artifacts during exercise. No actual AFib confirmed.

**The challenge**: In screening (prevalence ~0.1%), even 98% specificity yields 20:1 false positive ratio.

---

In [ ]:
# Import librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import signalfrom scipy.stats import kurtosis, skewfrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import classification_report, confusion_matrix, roc_auc_scoreimport warningswarnings.filterwarnings('ignore')np.random.seed(42)print("Libraries imported successfully!")

## 1. Generate Synthetic ECG DataWe'll create synthetic ECG waveforms for normal sinus rhythm and atrial fibrillation.

In [ ]:
def generate_synthetic_ecg(duration=30, fs=250, rhythm='normal'):    """    Generate synthetic ECG waveform.    Parameters:    - duration: seconds    - fs: sampling frequency (Hz)    - rhythm: 'normal' or 'afib'    """    t = np.arange(0, duration, 1/fs)    if rhythm == 'normal':        # Normal sinus rhythm: regular R-R intervals        hr = 70  # bpm        rr_interval = 60 / hr        n_beats = int(duration / rr_interval)        ecg = np.zeros_like(t)        for i in range(n_beats):            beat_time = i * rr_interval            beat_idx = int(beat_time * fs)            if beat_idx + 50 < len(ecg):                # Simplified QRS complex                ecg[beat_idx:beat_idx+10] = np.linspace(0, 1, 10)                ecg[beat_idx+10:beat_idx+20] = np.linspace(1, -0.3, 10)                ecg[beat_idx+20:beat_idx+30] = np.linspace(-0.3, 0, 10)        # Add P and T waves        ecg = signal.filtfilt(*signal.butter(3, 0.5, fs=fs), ecg)    else:  # afib        # AFib: irregular R-R intervals, no P waves        hr_mean = 90        hr_std = 20        rr_intervals = np.random.normal(60/hr_mean, 60/hr_std, 100)        rr_intervals = np.abs(rr_intervals)  # No negative intervals        ecg = np.zeros_like(t)        current_time = 0        for rr in rr_intervals:            if current_time >= duration:                break            beat_idx = int(current_time * fs)            if beat_idx + 30 < len(ecg):                ecg[beat_idx:beat_idx+10] = np.linspace(0, 1, 10)                ecg[beat_idx+10:beat_idx+20] = np.linspace(1, -0.3, 10)                ecg[beat_idx+20:beat_idx+30] = np.linspace(-0.3, 0, 10)            current_time += rr        # Add baseline irregularity        ecg += np.random.normal(0, 0.02, len(ecg))    # Add noise    ecg += np.random.normal(0, 0.05, len(ecg))    return t, ecg# Generate examplest_normal, ecg_normal = generate_synthetic_ecg(duration=10, rhythm='normal')t_afib, ecg_afib = generate_synthetic_ecg(duration=10, rhythm='afib')# Plotfig, axes = plt.subplots(2, 1, figsize=(14, 8))axes[0].plot(t_normal, ecg_normal, 'b-', linewidth=1)axes[0].set_title('Normal Sinus Rhythm', fontsize=14, fontweight='bold')axes[0].set_ylabel('Amplitude (mV)')axes[0].set_xlim([0, 10])axes[0].grid(True, alpha=0.3)axes[1].plot(t_afib, ecg_afib, 'r-', linewidth=1)axes[1].set_title('Atrial Fibrillation', fontsize=14, fontweight='bold')axes[1].set_xlabel('Time (seconds)')axes[1].set_ylabel('Amplitude (mV)')axes[1].set_xlim([0, 10])axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Key differences:")print("  Normal: Regular R-R intervals, visible P waves")print("  AFib: Irregular R-R intervals, absent P waves, irregular baseline")

## 2. Feature-Based ClassificationTraditional approach: Extract hand-crafted features from ECG signal.

In [ ]:
def extract_rr_intervals(ecg, fs=250):    """Detect R-peaks and calculate R-R intervals."""    # Simple peak detection    threshold = np.mean(ecg) + 2 * np.std(ecg)    peaks, _ = signal.find_peaks(ecg, height=threshold, distance=fs*0.4)    rr_intervals = np.diff(peaks) / fs  # Convert to seconds    return peaks, rr_intervalsdef extract_hrv_features(rr_intervals):    """Calculate heart rate variability features."""    if len(rr_intervals) < 2:        return {}    features = {        'rr_mean': np.mean(rr_intervals),        'rr_std': np.std(rr_intervals),        'rr_rmssd': np.sqrt(np.mean(np.diff(rr_intervals)**2)),        'rr_min': np.min(rr_intervals),        'rr_max': np.max(rr_intervals),        'rr_range': np.max(rr_intervals) - np.min(rr_intervals),        'hr_mean': 60 / np.mean(rr_intervals),        'hr_std': np.std(60 / rr_intervals),    }    return features# Extract features from examplespeaks_normal, rr_normal = extract_rr_intervals(ecg_normal)peaks_afib, rr_afib = extract_rr_intervals(ecg_afib)features_normal = extract_hrv_features(rr_normal)features_afib = extract_hrv_features(rr_afib)print("Normal Sinus Rhythm Features:")for k, v in features_normal.items():    print(f"  {k}: {v:.3f}")print("\nAtrial Fibrillation Features:")for k, v in features_afib.items():    print(f"  {k}: {v:.3f}")print("\n💡 Notice: AFib has higher variability (rr_std, rr_rmssd)")

## 3. Generate Dataset and Train ClassifierCreate a larger dataset and train Random Forest on HRV features.

In [ ]:
# Generate datasetn_samples = 500X_features = []y_labels = []print("Generating dataset...")for i in range(n_samples):    rhythm = 'normal' if i < n_samples // 2 else 'afib'    label = 0 if rhythm == 'normal' else 1    _, ecg = generate_synthetic_ecg(duration=30, rhythm=rhythm)    peaks, rr = extract_rr_intervals(ecg)    features = extract_hrv_features(rr)    if len(features) > 0:        X_features.append(list(features.values()))        y_labels.append(label)X = np.array(X_features)y = np.array(y_labels)feature_names = list(features_normal.keys())print(f"Dataset shape: {X.shape}")print(f"Features: {feature_names}")print(f"Class distribution: {np.bincount(y)}")# Split dataX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)# Train Random Forestrf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)rf.fit(X_train, y_train)# Evaluatey_pred = rf.predict(X_test)y_pred_proba = rf.predict_proba(X_test)[:, 1]print("\nPerformance on Test Set:")print(classification_report(y_test, y_pred, target_names=['Normal', 'AFib']))auroc = roc_auc_score(y_test, y_pred_proba)print(f"AUROC: {auroc:.3f}")# Confusion matrixcm = confusion_matrix(y_test, y_pred)plt.figure(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=['Normal', 'AFib'], yticklabels=['Normal', 'AFib'])plt.ylabel('True Label')plt.xlabel('Predicted Label')plt.title('Confusion Matrix: Feature-Based Classification')plt.show()

## 4. The Prevalence ProblemEven with 98% specificity, screening low-risk populations yields poor PPV.

In [ ]:
def calculate_ppv(sensitivity, specificity, prevalence):    """Calculate positive predictive value."""    tp = sensitivity * prevalence    fp = (1 - specificity) * (1 - prevalence)    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0    fp_ratio = fp / tp if tp > 0 else float('inf')    return ppv, fp_ratio# From confusion matrixtn, fp, fn, tp = cm.ravel()sensitivity = tp / (tp + fn)specificity = tn / (tn + fp)print(f"Model Performance (clinical setting):")print(f"  Sensitivity: {sensitivity:.1%}")print(f"  Specificity: {specificity:.1%}")# Calculate PPV for different settingssettings = [    ('Hospital (symptomatic)', 0.10),    ('Screening (general)', 0.005),    ('Yuki (young, fit)', 0.001)]print("\n" + "="*70)print("POSITIVE PREDICTIVE VALUE BY SETTING")print("="*70)for setting, prev in settings:    ppv, fp_ratio = calculate_ppv(sensitivity, specificity, prev)    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")print("="*70)print("\n⚠️  CLINICAL INSIGHT:")print("   Same model, different settings → vastly different clinical utility")print("   For Yuki (prevalence 0.1%), even 98% specificity → 20 false alarms per true AFib!")

## 5. Motion ArtifactsWearable devices are especially susceptible to motion artifacts during exercise.

In [ ]:
def add_motion_artifact(ecg, intensity=0.5):    """Add motion artifact to ECG signal."""    artifact = intensity * np.random.randn(len(ecg))    artifact = signal.filtfilt(*signal.butter(2, 0.05), artifact)    return ecg + artifact# Generate normal ECG with motion artifactt_motion, ecg_clean = generate_synthetic_ecg(duration=10, rhythm='normal')ecg_motion = add_motion_artifact(ecg_clean, intensity=0.3)# Comparefig, axes = plt.subplots(3, 1, figsize=(14, 10))axes[0].plot(t_motion, ecg_clean, 'b-', linewidth=1)axes[0].set_title('Clean ECG (Normal Sinus Rhythm)', fontsize=12, fontweight='bold')axes[0].set_ylabel('Amplitude')axes[0].grid(True, alpha=0.3)axes[1].plot(t_motion, ecg_motion, 'orange', linewidth=1)axes[1].set_title('ECG with Motion Artifact (e.g., during exercise)', fontsize=12, fontweight='bold')axes[1].set_ylabel('Amplitude')axes[1].grid(True, alpha=0.3)axes[2].plot(t_afib, ecg_afib, 'r-', linewidth=1)axes[2].set_title('True AFib', fontsize=12, fontweight='bold')axes[2].set_xlabel('Time (seconds)')axes[2].set_ylabel('Amplitude')axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("⚠️  Observation: Motion artifact can mimic AFib's irregular baseline!")print("   This is why wearable devices generate many false positives during exercise.")

## 6. Key Takeaways

### What We Learned

1. **ECG rhythm classification**: Can use hand-crafted features (HRV) or deep learning (1D CNN)

2. **Prevalence matters enormously**:
   - Hospital (10% AFib): PPV ~84% (good)
   - Screening (0.1% AFib): PPV ~4.6% (terrible)
   - Same model, vastly different utility

3. **Motion artifacts**: Major challenge for wearable devices
   - Exercise, arm movement mimics irregular rhythm
   - Leads to false positives

4. **Clinical implications**:
   - Wearable AFib detection useful for high-risk individuals
   - Should not be used for mass screening of low-risk populations
   - Need confirmatory testing pathways

5. **Yuki's experience** (Journey 2): Multiple false alarms caused anxiety, unnecessary testing, and ultimately loss of trust in technology

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Yuki's story provides the clinical motivation
- **Chapter 5 (Evaluation)**: Prevalence dependence of PPV, metrics in imbalanced settings
- **Chapter 7 (Time Series)**: This chapter - signal processing, frequency domain features, 1D CNNs
- **Chapter 11 (Deployment)**: Deployment considerations, user experience, regulatory pathways

### Real-World Devices

- Apple Watch, Fitbit, KardiaMobile: FDA-cleared for AFib detection
- Clinical guidelines: Use with confirmatory testing, not as standalone diagnostic
- Appropriate for: Patients with symptoms or known AFib for monitoring

---

## Exercises

1. Implement frequency domain features (FFT, power spectral density)
2. Build a 1D CNN for end-to-end classification from raw waveform
3. Simulate different prevalence settings and calculate required specificity for PPV >50%
4. Add varying levels of motion artifact and measure degradation

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 7: Time Series Analysis)*  
*This implements Journey 2 (Yuki - AFib Detection) from Chapter 3*  
*For clinical context, see Chapter 3. For time series methods, see Chapter 7.*